# World Happiness Report : Data Wrangling Pipeline

This notebook documents the journey of our dataset from raw data collection through to a validated, analysis-ready CSV.

**Structure:**
1. **Load Raw Data** : Load the Kaggle happiness data, add World Bank indicators, and merge into a single panel dataset.
2. **Data Structuring & Integration** : Merge, reshape, perform initial quality fixes, and drop redundant columns.
3. **Discovery & Profiling** : Examine shapes, types, distributions, and missingness before making any changes.
4. **Data Cleaning** : Handle missing values, remove duplicates.
5. **Data Structuring** : Enforce correct dtypes, reorder columns logically.
6. **Data Validation** : Range checks, final null audit, and summary.


---
---
# Part 0: Load Raw Data

The World Happiness Report CSV gives us country-year happiness scores, but we want to add socioeconomic indicators from the World Bank to gain more insights, so let's do that before we start wrangling.

**Steps:**
1. Load the happiness data and convert country names into ISO3 codes so we can use the World Bank API
2. Choose World Bank indicators across multiple themes (economy, health, education, governance, infrastructure, labour)
3. Fetch data from the World Bank API


## 0.1 Imports & Setup

`missing_report` gives a quick view of which columns have missing data and how bad it is.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import math
import country_converter as coco
from HZ_WH_helper_functions import missing_report
from HZ_WH_helper_functions import fetch_wb

pd.set_option('display.float_format', '{:.3f}'.format)

## 0.2 Load the Happiness Dataset

In [ ]:
happiness_df = pd.read_csv('drafts/datasets/world_happiness_report.csv')

print(f'Shape: {happiness_df.shape}')
print(f'Countries: {happiness_df["Country Name"].nunique()}')
happiness_df.head()

## 0.3 Convert Country Names -> ISO3 Codes

We need standardised country codes so we can join the happiness data with World Bank data. The `country_converter` library handles the messy name-matching for us.

In [ ]:
cc = coco.CountryConverter()

unique_countries = happiness_df['Country Name'].unique()
iso3_series = cc.pandas_convert(pd.Series(unique_countries), to='ISO3')

# 'not found' -> NaN so they don't break API calls later on
iso3_series = iso3_series.replace('not found', np.nan)

iso_map = dict(zip(unique_countries, iso3_series))
happiness_df['ISO3'] = happiness_df['Country Name'].map(iso_map)

not_found = happiness_df.loc[happiness_df['ISO3'].isna(), 'Country Name'].unique()
print(f'Countries not converted to ISO3: {len(not_found)}')
if len(not_found):
    print(not_found)

## 0.4 Define World Bank Indicators

We pull indicators across six themes that theoretically explain happiness:
- **Wealth & Economy** : income, inflation, trade
- **Health** : life expectancy, child mortality, health spending
- **Education** : expected years of schooling
- **Infrastructure** : internet, electricity, urbanisation
- **Labour & Social** : unemployment, female participation, population
- **Governance** (WGI) : corruption, rule of law, political stability

In [ ]:
WB_INDICATORS = {
    # --- Wealth & Economy ---
    'NY.GDP.PCAP.CD':       'GDP/Capita',
    'NY.GDP.MKTP.KD.ZG':    'GDP Growth Rate',
    'FP.CPI.TOTL.ZG':       'Inflation Rate',
    'NE.TRD.GNFS.ZS':       'Trade Openness',

    # --- Health ---
    'SP.DYN.LE00.IN':       'Life Expectancy',
    'SH.DYN.MORT':          'Child Mortality Rate',
    'SH.XPD.CHEX.GD.ZS':    'Health Expenditure pct of GDP',
    'SH.XPD.OOPC.CH.ZS':    'Out Of Pocket Health pct',

    # --- Education ---
    'SE.COM.DURS':          'Expected Years Of Schooling',

    # --- Infrastructure & Modernisation ---
    'IT.NET.USER.ZS':       'Internet Penetration',
    'EG.ELC.ACCS.ZS':       'Electricity Access pct',
    'SP.URB.TOTL.IN.ZS':    'Urbanization Rate',
    'IS.ROD.PAVE.ZS':       'Paved Roads pct',

    # --- Labour & Social ---
    'SL.UEM.TOTL.ZS':       'Unemployment Rate',
    'SL.TLF.CACT.FE.ZS':    'Female Labor Participation',
    'SP.POP.TOTL':          'Population',
    'SP.POP.GROW':          'Population Growth Rate',
}

# Governance indicators come from a different World Bank database (WGI = db 3)
WGI_INDICATORS = {
    'CC.EST':  'Control Of Corruption',
    'GE.EST':  'Govt Effectiveness',
    'RL.EST':  'Rule Of Law',
    'VA.EST':  'Voice Accountability',
    'PV.EST':  'Political Stability',
    'RQ.EST':  'Regulatory Quality',
}

print(f'Total WDI indicators: {len(WB_INDICATORS)}')
print(f'Total WGI indicators: {len(WGI_INDICATORS)}')

## 0.5 Fetch World Bank Data

The World Bank API is slow; a full fetch takes 10-30 minutes and usually fails mid-fetch. 

Rather than make this notebook grind to a halt, we ran the fetch separately using `fetch_world_bank.py`, and saved the raw outputs to `wdi_raw.csv` and `wgi_raw.csv`.

Below is the fetch code for reference (commented out). The cell that actually runs loads the pre-fetched CSVs is below it.

> To re-fetch from scratch: go to `HZ_WH_helper_functions.py`, read its instructions, run it, then re-run this notebook.

In [ ]:
# this does work, but it takes forever to run, so i suggest you work with the already imported csv's in the cell below

# years = sorted(happiness_df['Year'].unique())
# iso3_list = happiness_df['ISO3'].dropna().unique().tolist()
# wdi_keys = list(WB_INDICATORS.keys())
# wgi_keys = list(WGI_INDICATORS.keys())
#
# print(f'Fetching data for {len(iso3_list)} countries, years {years[0]}-{years[-1]}')
#
# df_wdi = fetch_wb(iso3_list, years, wdi_keys, batch_size=6)
# df_wgi = fetch_wb(iso3_list, years, wgi_keys, db_type=2)
#
# df_wdi.to_csv('drafts/datasets/wdi_raw.csv', index=False)
# df_wgi.to_csv('drafts/datasets/wgi_raw.csv', index=False)

In [ ]:
# -- Load the pre-fetched World Bank data --
df_wdi = pd.read_csv('drafts/datasets/wdi_raw.csv')
df_wgi = pd.read_csv('drafts/datasets/wgi_raw.csv')

print(f'WDI: {df_wdi.shape[0]:,} rows * {df_wdi.shape[1]} cols')
print(f'WGI: {df_wgi.shape[0]:,} rows * {df_wgi.shape[1]} cols')
df_wdi.head()

---
---
# Part 1: Data Structuring & Integration

With the raw data loaded, we now merge and reshape.

**Steps:**
1. Merge & reshape the World Bank data from wide to long format and join with the happiness data
2. Fill in missing `Regional Indicator` values
3. Drop `Log GDP Per Capita`; it is redundant given `GDP/Capita` from the World Bank


## 1.1 Merge & Reshape

The World Bank API returns data in wide format (`ISO3 | series/indicator | YR2015 | YR2016 | ...`), but it'll be easier to work with if the columns were the indicators. We will fix the wbgapi dfs' format, then merge with the happiness data to create the df we will be working with for the rest of the notebook.


In [ ]:
def melt_and_pivot(df):
    # identify year columns
    year_cols = [c for c in df.columns if c not in ('ISO3', 'series')]
    
    # Unpivot Years to become one column
    long = df.melt(id_vars=['ISO3', 'series'], value_vars=year_cols, var_name='Year', value_name='value')
    
    # Normalise year strings: 'YR2015' or '2015' -> int
    long = long.assign(Year=lambda x: x['Year'].str.extract(r'(\d+)', expand=False).astype(int))
    
    # the indicator codes in the series column become column names
    pivot = (
        long.pivot_table(index=['ISO3', 'Year'], columns='series',
                         values='value', aggfunc='first').reset_index()
                         .rename_axis(columns=None)
    )
    return pivot


wdi_pivot = melt_and_pivot(df_wdi)
wgi_pivot = melt_and_pivot(df_wgi)
df_wb = pd.merge(wdi_pivot, wgi_pivot, on=['ISO3', 'Year'], how='outer')

# wb data returns with indicator codes as column names instead of human-readable labels, 
# so let's rename the columns using the mapping dicts we already have
rename_map = {**WB_INDICATORS, **WGI_INDICATORS}
df_wb.rename(columns=rename_map, inplace=True)

# Merge with happiness data
df = pd.merge(happiness_df, df_wb, on=['ISO3', 'Year'], how='left')

print(f'Merged shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head()

## 1.2 Initial Quality Fix: Region Assignment

Some countries in the happiness dataset are missing their `Regional Indicator`. Since this is a metadata field that we know the correct values for, we fix it here at creation time rather than during the cleaning stage.


In [ ]:
regions_missing_count = df['Regional Indicator'].isnull().sum()
regions_missing = df[df['Regional Indicator'].isnull()]
missing_countries = regions_missing['Country Name'].unique()

print(f"Countries with missing regions: {len(missing_countries)}")
print(f"List of countries: {missing_countries}")
print(f"Total rows missing a region: {regions_missing_count}")

In [ ]:
# Fill missing Regional Indicator values using a manual lookup
manual_region_map = {
    "Angola": "Sub-Saharan Africa",
    "Belize": "Latin America and Caribbean",
    "Bhutan": "South Asia",
    "Central African Republic": "Sub-Saharan Africa",
    "Congo (Kinshasa)": "Sub-Saharan Africa",
    "Cuba": "Latin America and Caribbean",
    "Czechia": "Central and Eastern Europe",
    "Djibouti": "Sub-Saharan Africa",
    "Guyana": "Latin America and Caribbean",
    # WB classifies Israel as Middle East and North Africa, but it's more commonly grouped with Western Europe in other datasets
    "Israel": "Western Europe",  
    "Oman": "Middle East and North Africa",
    "State of Palestine": "Middle East and North Africa",
    "Qatar": "Middle East and North Africa",
    "Sudan": "Sub-Saharan Africa",
    "Somaliland region": "Sub-Saharan Africa",
    "Somalia": "Sub-Saharan Africa",
    "South Sudan": "Sub-Saharan Africa",
    "Suriname": "Latin America and Caribbean",
    "Eswatini": "Sub-Saharan Africa",
    "Syria": "Middle East and North Africa",
    "Trinidad and Tobago": "Latin America and Caribbean",
    "Turkiye": "Middle East and North Africa",
}

df['Regional Indicator'] = df.apply(
    lambda row: manual_region_map.get(row['Country Name'], row['Regional Indicator']),
    axis=1 # so it iterates over rows, not columns
)

regions_missing = df['Regional Indicator'].isnull().sum()
print(f"Regions still missing after manual assignment: {regions_missing}")
if regions_missing > 0:
    print("Countries still missing:", df[df['Regional Indicator'].isnull()]['Country Name'].unique())

In [ ]:
# Rename and consolidate regions
region_rename = {
    "Commonwealth of Independent States": "Post-Soviet Bloc",
    "Western Europe": "West EU & CANZUK & USA",
    "North America and ANZ": "West EU & CANZUK & USA",
}
df["Regional Indicator"] = df["Regional Indicator"].replace(region_rename)
print("Regions after renaming:")
print(sorted(df["Regional Indicator"].unique()))

## 1.3 Drop Redundant Column:

The happiness dataset includes `Log GDP Per Capita`, and we have also pulled `GDP/Capita` from the World Bank API, so we'll keep `GDP/Capita` because the raw dollar value is more interpretable, and log transformations can always be applied later.


In [ ]:
# Drop Log GDP Per Capita â€” redundant with GDP/Capita from the World Bank
if 'Log GDP Per Capita' in df.columns:
    df = df.drop(columns=['Log GDP Per Capita'])
    print("Dropped 'Log GDP Per Capita'. Remaining columns:", df.shape[1])
else:
    print("'Log GDP Per Capita' not found â€” already absent.")


---
# Part 2: Data Discovery & Profiling

Now that the dataset exists, we profile it *before touching anything*. This answers:
- How many rows and columns?
- What are the data types? Are the numeric columns actually numeric?
- Which columns have missing values, and how severe is it?
- What do the distributions look like, how are the outliers & tails?

We observe and document here. No modifications.

In [ ]:
# loot at shape and dtypes
print(f"Shape: {df.shape[0]:,} rows * {df.shape[1]} columns")
print(f"\nColumn names and dtypes:")
print(df.dtypes.to_string())
df.head()

# all seems to be in order

In [ ]:
# Countries per region
region_counts = df.groupby('Regional Indicator')['Country Name'].nunique().sort_values(ascending=False)
print('Countries per region:')
print(region_counts.to_string())
print(f"Total: {df['Country Name'].nunique()} countries across {df['Regional Indicator'].nunique()} regions")

In [ ]:
# Statistical summary
print(df.describe())

"""
A few observations here:
* Genorosity's mean is 0, which means that people are donating roughly as much as you'd expect given how rich their country is
* Most people around the world think their government is corrupt
* A few massive countries like China and India are skewing population statistics, maing the mean much higher than the median
* For Healthy life expectancy, the lowest value in the data (6.72 years) seems suspiciously wrong
* The variables with the most gaps (Generosity, Corruption, Life Expectancy) will be checked in step 2 if the missing entries 
  are scattered or clustered in particular countries or time periods.
"""

In [ ]:
# After seeing the variations between counts in the columns, Missing values report
print(f"Columns with missing values: {df.isnull().any().sum()} of {df.shape[1]}")
print(f"Total missing cells: {df.isnull().sum().sum():,}\n")
missing_report(df)

In [ ]:
# Histograms for all numeric columns
num_cols = df.select_dtypes(include='number').columns.tolist()
graphs_per_row = 4
row_count = math.ceil(len(num_cols) / graphs_per_row)

fig, axes = plt.subplots(row_count, graphs_per_row, figsize=(20, row_count * 3))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    optimal_bins = min(30, len(np.histogram_bin_edges(df[col].dropna(), bins='fd')) - 1)
    axes[i].hist(df[col].dropna(), bins=optimal_bins, color='steelblue', edgecolor='white', linewidth=0.4)
    axes[i].set_title(col, fontsize=9)
    axes[i].tick_params(labelsize=8)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distributions of All Numeric Columns (raw data)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

### Distribution Insights
- **Inflation Rate, GDP/Capita, Population, Child Mortality Rate** - and other long-tailed variables will need to be log transformed before modeling
- **Internet Penetration, Female Labor Participation, Urbanization Rate** - bimodal, reflecting the developed/developing world divide; may model separately
- **Electricity Access** - massive spike at 100%, limits its usefulness as a coefficient
- **GDP Growth Rate** - extreme outliers on both ends, consider winsorizing
- **Expected Years of Schooling** - stepped/chunky distribution, worth checking source methodology
- **Perceptions of Corruption** - skews heavily toward high values, as previously noted



---
# Part 3: Data Cleaning

Based on the profiling above, we apply the following cleaning steps in order:

1. **Remove duplicate rows** as duplicates  introduce biases to the analysis and future models.
2. **Per-country linear interpolation**, fills interior gaps using each country's own time series, with forward/backward fill for edge years.
3. **Drop remaining rows with nulls** - any rows that interpolation couldn't fill are removed.

> **Limitation:** Linear interpolation assumes smooth change between known data points. This is reasonable for slow-moving indicators (life expectancy, urbanisation) but may understate volatility in fast-changing ones like inflation rate or GDP growth. We acknowledge this trade-off, but alternatives like mean imputation would ignore the country's trajectory entirely, which is arguably worse for the data's accuracy.

In [ ]:
# Remove duplicate rows
n_dupes = df.duplicated().sum()
print(f"Duplicate rows found: {n_dupes}")
if n_dupes > 0:
    df = df.drop_duplicates()
    print(f"Removed. Remaining rows: {len(df):,}")

print()
# Missing data snapshot
print(f"Rows with at least one missing value: {df.isnull().any(axis=1).sum()} of {len(df)}")
print()
missing_report(df)

In [ ]:
# Per-country linear interpolation for numeric columns
# This uses each country's own time series to fill gaps - better than a global statistic
# for panel data, though it assumes smooth transitions between known points.
df = df.sort_values(['ISO3', 'Year']).reset_index(drop=True)

interp_cols = [
    c for c in df.select_dtypes(include='number').columns
    if c != 'Year'
]

df[interp_cols] = (
    df.groupby('ISO3')[interp_cols]
      .transform(lambda g: g.interpolate(method='linear').ffill().bfill())
)

print()
# Missing data snapshot
print(f"Rows with at least one missing value: {df.isnull().any(axis=1).sum()} of {len(df)}")
print()
missing_report(df)

In [ ]:
# Drop rows that still have missing values after interpolation
before = len(df)
df = df.dropna()
dropped = before - len(df)
if dropped > 0:
    print(f"Dropped {dropped} row(s) with remaining nulls.")
else:
    print("No rows dropped - all values filled.")
print(f"Shape after cleaning: {df.shape}")

print()
# Missing data snapshot
print(f"Rows with at least one missing value: {df.isnull().any(axis=1).sum()} of {len(df)}")

---
# Part 4: Data Structuring

With cleaning complete, we enforce correct data types and arrange columns in a logical order for downstream analysis.

- **dtype enforcement** : `Year` is cast to `int`.
- **column reordering** : identifiers â†’ happiness metrics â†’ World Bank indicators.


In [ ]:
# Ensure Year is integer
df['Year'] = df['Year'].astype(int)
print(f"Year dtype: {df['Year'].dtype}")
print(f"Year range: {df['Year'].min()} - {df['Year'].max()}")

In [ ]:
# Reorder columns: identifiers -> happiness metrics -> World Bank indicators
id_cols = ['Country Name', 'ISO3', 'Regional Indicator', 'Year']

happiness_cols = [
    'Life Ladder', 'Social Support',
    'Healthy Life Expectancy At Birth', 'Freedom To Make Life Choices',
    'Generosity', 'Perceptions Of Corruption', 'Positive Affect',
    'Negative Affect', 'Confidence In National Government'
]

# Everything else is a World Bank indicator
wb_cols = [c for c in df.columns if c not in id_cols + happiness_cols]

# Order the columns
ordered = [c for c in id_cols + happiness_cols + wb_cols if c in df.columns]
df = df[ordered]

print(f"Final column order ({len(ordered)} columns):")
print(f"  Identifiers    : {id_cols}")
print(f"  Happiness cols : {[c for c in happiness_cols if c in df.columns]}")
print(f"  World Bank cols: {wb_cols}")

---
# Part 5: Data Validation

Cleaning alone isn't enough - we verify the data makes sense before analysis.

Checks:
- **Value ranges**: `Life Ladder` is a 0-10 Cantril scale; percentages must be 0-100; survey columns are 0-1.
- **Final shape and summary**: a clean snapshot of what the pipeline produced.

We don't re-check for nulls here since we explicitly dropped all rows with missing values at the end of Part 3. Out-of-range values don't necessarily mean errors - they could be legitimate edge cases or data-collection artefacts. We flag and document them, not blindly discard.

In [ ]:
def check_range(dataframe, col, lo, hi):
    """Print a one-line report on values outside the expected [lo, hi] range."""
    if col not in dataframe.columns:
        print(f"  {col}: column not found (may have been dropped)")
        return
    out = dataframe[(dataframe[col] < lo) | (dataframe[col] > hi)]
    status = "OK" if len(out) == 0 else f"*** {len(out)} out-of-range value(s) ***"
    print(f"  {col:<35} expected [{lo}, {hi:>3}]  ->  {status}")

print("Range checks:")
print()

check_range(df, 'Life Ladder', 0, 10)

print()
print("  -- Percentage columns (0-100) --")
for col in ['Electricity Access pct', 'Internet Penetration', 'Unemployment Rate',
            'Female Labor Participation', 'Health Expenditure pct of GDP',
            'Out Of Pocket Health pct', 'Urbanization Rate']:
    check_range(df, col, 0, 100)

print()
print("  -- Survey scale columns (0-1) --")
for col in ['Social Support', 'Freedom To Make Life Choices',
            'Perceptions Of Corruption', 'Positive Affect', 'Negative Affect']:
    check_range(df, col, 0, 1)

In [ ]:
# Final summary
print("=" * 55)
print("  CLEANED DATASET SUMMARY")
print("=" * 55)
print(f"  Shape        : {df.shape[0]:,} rows * {df.shape[1]} columns")
print(f"  Years        : {df['Year'].min()} - {df['Year'].max()}")
print(f"  Countries    : {df['Country Name'].nunique()}")
print(f"  Regions      : {df['Regional Indicator'].nunique()}")
print(f"  Missing cells: {df.isnull().sum().sum()}")
print()
print("  Column dtypes:")
for col, dtype in df.dtypes.items():
    print(f"    {col:<45} {dtype}")

---
# Save Cleaned Dataset

In [ ]:
df.to_csv('cleaned-datasets/HZ_WH_data.csv', index=False)
print("Saved cleaned dataset to cleaned-datasets/HZ_WH_data.csv")